In [5]:
import os
from tqdm import tqdm
import json
from nltk.tokenize import word_tokenize
import random

def open_data(path):
    "Open dataset at path location and read the data into a list."
    with open(path, "r", encoding="utf-8") as f:
        data = [line.strip().split("\t") for line in f.readlines()]
    return data

# Extract questions with indirect answers from OPUS de-en
Applies light pre-filtering by searching questions followed by lines without direct answer particles in the German subtitles

In [ ]:
# search the OPUS corpus for more aligned indirect QA-pairs by filtering the English data and checking manually

def search_opus_indirect_qa_pairs(opus_dir_path_src, k, opus_dir_path_tgt, qa_pairs=None):
    """Searches QA pairs in a randomly sampled subset (k%) the given OPUS directory and writes them to a file. 
    If given, retrieves parallel translations and considers existing QA data.

    @args:
        opus_dir_path_src: directory with OPUS files in source language
        k: optional; percentage of OPUS files to consider
        opus_dir_path_tgt: optional; directory with OPUS files in target language to retrieve translations from
        qa_pairs: optional; list with (questions, answer, label) tuples to consider (filter)

    @returns:
        src_qa_pairs: list of (question, answer) tuples in src language
        tgt_qa_pairs: optinal; list of (question, answer) tuples in tgt language
    """
    src_qa_pairs = []
    tgt_qa_pairs = []

    direct_answer_particles = {"Ja", "ja", "Nein", "nein"}

    found = [] # prevents duplicates
    if qa_pairs != None:
        found += [q for q, _, _ in qa_pairs] # add already obtained qa data if given

    # iterate over all opensubtitles file (split due to enormous data size)
    for _, _, file_names in os.walk(opus_dir_path_src): 
        # randomly sample k% of the files
        k = round(len(file_names) * (k/100))
        file_names = random.sample(file_names, k)

        for file_name in tqdm(file_names, total=len(file_names), desc="Iteration over OPUS files... "): 
            # read content of current OPUS file
            file_path_src = os.path.join(opus_dir_path_src, file_name)
            with open(file_path_src, "r", encoding="utf-8") as f:
                data_src = [line.strip() for line in f.readlines()]
            
            # read files of parallel OPUS file in tgt language
            file_name_tgt = f"{file_name[:19]}.{opus_dir_path_tgt[-2:]}.{file_name[23:]}"
            file_path_tgt = os.path.join(opus_dir_path_tgt, file_name_tgt)
            with open(file_path_tgt, "r", encoding="utf-8") as f:
                data_tgt = [line.strip() for line in f.readlines()]
            
            # iterate over OPUS data and compare to opensubtitles data
            for sent_id, sent in enumerate(data_src[:-1]):
                # filter for questions and duplicates 
                if sent.endswith("?") and sent not in found:
                    answer = data_src[sent_id+1]
                    answer_tokens = set(word_tokenize(answer))

                    # check that answer does not contain direct answer particles 
                    if not direct_answer_particles & answer_tokens: 
                        # retrieve translation
                        sent_tgt = data_tgt[sent_id]

                            # check that translation is also a question and are not Wh-questions with initial question words
                        if sent_tgt.endswith("?") and not sent_tgt.startswith("Wh"): 
                            answer_tgt = data_tgt[sent_id+1]

                            src_qa_pairs.append((sent, answer))
                            tgt_qa_pairs.append((sent_tgt, answer_tgt))

                            # prevents duplicate QA pairs
                            found.append(sent)

    return src_qa_pairs, tgt_qa_pairs


# extract German QA pairs that are not already in IndirectQA
# IQA pairs from IndirectQA that also appear in opensubtitles de
indirectqa_de_com_de = open_data("opensubtitles/IndirectQA_original_de_com.tsv")
indirectqa_de_crime_de = open_data("opensubtitles/IndirectQA_original_de_crime.tsv")

# english and german opensubtitles data
# available at https://opus.nlpl.eu/OpenSubtitles/en&de/v2024/OpenSubtitles
opus_de_en_dir_en = "opensubtitles/de-en.en"
opus_de_en_dir_de = "opensubtitles/de-en.de"

# extract viable candidates
opus_de_candidates_de, opus_de_candidates_en = search_opus_indirect_qa_pairs(opus_de_en_dir_de, 1, opus_de_en_dir_en, indirectqa_de_com_de + indirectqa_de_crime_de)

# store extracted candiates in json files
path = "opensubtitles/opus_de_candidates_de.json"
with open(path, "a", encoding="utf-8") as f:
    json.dump(opus_de_candidates_de, f, indent=2)

path = "opensubtitles/opus_de_candidates_en.json"
with open(path, "a", encoding="utf-8") as f:
    json.dump(opus_de_candidates_en, f, indent=2)

### Filter candidates by genre

In [ ]:
# opus sentences per genre
with open("opensubtitles/opus_de_raw_genre_to_questions.json", "r", encoding="utf-8") as f:
    genre_to_questions = json.load(f)

opus_comedy = list(set(genre_to_questions["Comedy"]))
opus_crime = list(set(genre_to_questions["Crime"]))

opus_de_com_de = []
opus_de_com_en = []
opus_de_crime_de = []
opus_de_crime_en = []

# randomly sample k% of candidates
candidates = list(zip(opus_de_candidates_de, opus_de_candidates_en))

k = 30
k = round(len(candidates) * (k/100))
candidates = random.sample(candidates, k)

for (q_de, a_de), (q_en, a_en) in tqdm(candidates, total=len(candidates)):
    if q_de in opus_comedy:
        opus_de_com_de.append((q_de, a_de))
        opus_de_com_en.append((q_en, a_en))
    
    if q_de in opus_crime:
        opus_de_crime_de.append((q_de, a_de))
        opus_de_crime_en.append((q_en, a_en))

opus_de_en_com = set(list(zip(opus_de_com_en, opus_de_com_de)))
opus_de_en_crime = set(list(zip(opus_de_crime_en, opus_de_crime_de)))

opus_com_filtered = [item for item in opus_de_en_com if item not in opus_de_en_com&opus_de_en_crime]
opus_crime_filtered = [item for item in opus_de_en_com if item not in opus_de_en_com&opus_de_en_crime]


# store sorted candiates in json files
path = "opensubtitles/opus_en_de_candidates_com.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump(opus_com_filtered, f, indent=2)

path = "opensubtitles/opus_en_de_candidates_crime.json"
with open(path, "a", encoding="utf-8") as f:
    json.dump(opus_crime_filtered, f, indent=2)